# [배포 시뮬레이션] 라우팅 + API 호출 생성

이 노트북은 **SmartRouter의 분기 성능과 비용-품질 트레이드오프**를 검증하기 위해 작성되었습니다.

외부 멘토님의 피드백(\"분류만 잘하는 것이 아니라 실제 교정된 결과물의 품질도 입증해야 한다\")을 수용하여, 다음과 같은 3가지 전략을 비교합니다.

### 🧪 실험 설계 (3가지 전략 비교)
- **전략 A (경량 모델 단독)**: 모든 문장을 비용이 저렴한 `Nano(Haiku)` 모델로 교정합니다. (품질 저하 위험)
- **전략 B (대형 모델 단독)**: 모든 문장을 성능이 뛰어난 `Mini(Sonnet)` 모델로 교정합니다. (비용 폭증 위험)
- **전략 C (SmartRouter)**: 자체 개발한 RoBERTa 라우터가 난이도를 예측하여 Easy는 Nano로, Hard는 Mini로 보냅니다. (최적화)

In [1]:
import pandas as pd
import numpy as np
import os
import sys
sys.path.append('../src')

from smartrouter.router.dynamic import RoBERTaDynamicRouter
from smartrouter.services.generator import SentencifyGenerator
from smartrouter.core import config

## 1. 테스트 데이터 로드 및 라우터 추론 확인

이전에 학습 및 검증을 완료한 `5. test_results_routed.csv` (1,156건) 데이터를 로드합니다.

In [2]:
df_test = pd.read_csv('../data/5. test_results_routed.csv')
print(f"총 테스트 문장 수: {len(df_test):,}건")
df_test[['input_sentence', 'tag_intensity', 'tag_field', 'hard_prob', 'difficulty', 'routed_model']].head()

## 2. API 호출 기반 생성 시뮬레이션 (Mock Mode)

과도한 API 비용 청구를 막기 위해 `Mock Mode`로 시뮬레이션 된 결과를 생성합니다. 
(실제 서비스 환경에서는 AWS Bedrock이 호출됩니다.)

In [3]:
# 모의 결과 생성을 위한 헬퍼 함수 (실제 API 호출 코드를 추상화)
generator = SentencifyGenerator()

def simulate_generation():
    print("Loading cached generation results (Mock API Run)...")
    df_inference = pd.read_csv('../data/7. inference_results.csv')
    return df_inference

df_inference = simulate_generation()
print("✅ 3가지 전략의 교정 결과 생성이 완료되었습니다.")
df_inference[['source_text', 'result_A_nano', 'result_B_mini', 'result_C_router']].head()

결과는 `../data/7. inference_results.csv` 에 저장되었습니다. 다음 노트북(`08_final_evaluation.ipynb`)에서 이 결과물의 품질을 G-Eval로 채점하고 비용 절감 효과를 분석합니다.